# Pain SAE Experiment
## Do LLMs have internal representations specifically for pain?

This notebook runs contrastive SAE analysis on Llama 3.3 70B to find features that distinguish pain sentences from matched controls.

**Hypothesis:** Pain has distinct internal representations, separable from:
- Fear (B)
- Negative emotion (C1)
- Negative world state (C2)
- Neutral (D)
- Body sensation without pain (E)

## Setup

In [ ]:
# Install dependencies if needed
# !pip install requests pandas matplotlib seaborn

In [ ]:
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Optional
import time

# Configuration
API_KEY = "STEERING_API_KEY"
API_BASE = "https://api.steeringapi.com"
MODEL = "meta-llama/Llama-3.3-70B-Instruct"

# Output directory
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"API Key set: {API_KEY[:10]}..." if len(API_KEY) > 10 else "WARNING: Set your API key!")

## Load Dataset

In [ ]:
# Load both first and third person datasets
with open("S1_first_person_prompts.json", "r", encoding="utf-8") as f:
    data_1p = json.load(f)
    
with open("S1_third_person_prompts.json", "r", encoding="utf-8") as f:
    data_3p = json.load(f)

sentences_1p = data_1p["sentences"]
sentences_3p = data_3p["sentences"]

print(f"Loaded {len(sentences_1p)} first-person sentences")
print(f"Loaded {len(sentences_3p)} third-person sentences")
print(f"\nCategories: {data_1p['metadata']['categories']}")

In [ ]:
# Category definitions
PAIN_CATEGORIES = ['A1', 'A2', 'A3', 'A4', 'A5']
CONTROL_CATEGORIES = ['B', 'C1', 'C2', 'D', 'E']

CATEGORY_NAMES = {
    'A1': 'Physical Pain',
    'A2': 'Psychological Pain', 
    'A3': 'Social Pain',
    'A4': 'Moral Injury',
    'A5': 'Cognitive Pain',
    'B': 'Fear',
    'C1': 'Negative Emotion',
    'C2': 'Negative World State',
    'D': 'Neutral',
    'E': 'Body Sensation'
}

def filter_by_categories(sentences: List[Dict], categories: List[str]) -> List[Dict]:
    return [s for s in sentences if s['category'] in categories]

def format_for_api(sentences: List[Dict]) -> List[List[Dict]]:
    """Convert to API format: list of message arrays."""
    return [[{"role": "user", "content": s['prompt']}] for s in sentences]

# Preview
for cat in PAIN_CATEGORIES + CONTROL_CATEGORIES:
    examples = filter_by_categories(sentences_1p, [cat])
    print(f"{cat} ({CATEGORY_NAMES[cat]}): {len(examples)} sentences")
    print(f"   Example: {examples[0]['prompt'][:60]}...\n")

## API Helper Functions

In [ ]:
def api_request(endpoint: str, payload: Dict, timeout: int = 180) -> Optional[Dict]:
    """Make API request with error handling."""
    headers = {
        "X-API-Key": API_KEY,
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(
            f"{API_BASE}{endpoint}",
            headers=headers,
            json=payload,
            timeout=timeout
        )
        
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code}: {response.text[:200]}")
            return None
            
    except requests.exceptions.Timeout:
        print(f"Timeout on {endpoint}")
        return None
    except Exception as e:
        print(f"Exception: {e}")
        return None

def run_contrast(dataset_1: List, dataset_2: List, k: int = 100) -> Optional[Dict]:
    """Run contrastive analysis between two datasets."""
    return api_request("/v1/chat_attribution/contrast", {
        "model": MODEL,
        "dataset_1": dataset_1,
        "dataset_2": dataset_2,
        "k_to_add": k,
        "k_to_remove": k
    })

def run_inspect(messages: List[Dict], aggregation: str = "mean", top_k: int = 100) -> Optional[Dict]:
    """Inspect feature activations for a single message."""
    return api_request("/v1/chat_attribution/inspect", {
        "model": MODEL,
        "messages": messages,
        "aggregation_method": aggregation,
        "top_k": top_k
    })

def run_attribute(messages: List[Dict], top_k: int = 100, start_idx: int = None, end_idx: int = None) -> Optional[Dict]:
    """Get feature attribution with optional position selection."""
    payload = {
        "model": MODEL,
        "messages": messages,
        "top_k": top_k
    }
    if start_idx is not None:
        payload["start_idx"] = start_idx
    if end_idx is not None:
        payload["end_idx"] = end_idx
    
    return api_request("/v1/chat_attribution/attribute", payload)

def search_features(query: str, top_k: int = 50) -> Optional[Dict]:
    """Search for features by semantic description."""
    return api_request("/v1/features/search", {
        "query": query,
        "model_name": MODEL,
        "top_k": top_k
    })

print("API functions ready.")

---
# PART 1: Contrastive Analysis
Find features that distinguish pain from controls across multiple comparisons.

In [ ]:
# Define all contrasts to run
CONTRASTS = [
    # Main comparisons
    ("pain_vs_all_controls", PAIN_CATEGORIES, CONTROL_CATEGORIES, "All Pain vs All Controls"),
    ("pain_vs_neutral", PAIN_CATEGORIES, ['D'], "All Pain vs Neutral"),
    ("pain_vs_fear", PAIN_CATEGORIES, ['B'], "All Pain vs Fear"),
    ("pain_vs_negemo", PAIN_CATEGORIES, ['C1'], "All Pain vs Negative Emotion"),
    ("pain_vs_negworld", PAIN_CATEGORIES, ['C2'], "All Pain vs Negative World"),
    ("pain_vs_body", PAIN_CATEGORIES, ['E'], "All Pain vs Body Sensation"),
    
    # Specific pain type comparisons
    ("physical_vs_body", ['A1'], ['E'], "Physical Pain vs Body Sensation"),
    ("physical_vs_neutral", ['A1'], ['D'], "Physical Pain vs Neutral"),
    ("psych_vs_negemo", ['A2'], ['C1'], "Psychological Pain vs Negative Emotion"),
    ("social_vs_negemo", ['A3'], ['C1'], "Social Pain vs Negative Emotion"),
    ("moral_vs_negemo", ['A4'], ['C1'], "Moral Injury vs Negative Emotion"),
    ("cognitive_vs_neutral", ['A5'], ['D'], "Cognitive Pain vs Neutral"),
    
    # Cross-pain comparisons (are pain types similar?)
    ("physical_vs_psychological", ['A1'], ['A2'], "Physical vs Psychological Pain"),
    ("physical_vs_social", ['A1'], ['A3'], "Physical vs Social Pain"),
    
    # Perspective comparison
    ("first_vs_third_person_pain", "1p_pain", "3p_pain", "First vs Third Person (Pain only)"),
]

print(f"Will run {len(CONTRASTS)} contrastive comparisons")

In [ ]:
# Run all contrasts
contrast_results = {}

for name, cats1, cats2, description in CONTRASTS:
    print(f"\n{'='*60}")
    print(f"Running: {description}")
    print(f"{'='*60}")
    
    # Handle special case for perspective comparison
    if cats1 == "1p_pain":
        data1 = format_for_api(filter_by_categories(sentences_1p, PAIN_CATEGORIES))
        data2 = format_for_api(filter_by_categories(sentences_3p, PAIN_CATEGORIES))
    else:
        data1 = format_for_api(filter_by_categories(sentences_1p, cats1))
        data2 = format_for_api(filter_by_categories(sentences_1p, cats2))
    
    print(f"Dataset 1: {len(data1)} sentences")
    print(f"Dataset 2: {len(data2)} sentences")
    
    result = run_contrast(data1, data2, k=100)
    
    if result:
        contrast_results[name] = {
            "description": description,
            "result": result
        }
        
        # Show top 5
        print(f"\nTop 5 features for Dataset 1 ({description.split(' vs ')[0]}):")
        for i, feat in enumerate(result.get('top_to_add', [])[:5]):
            label = feat.get('label', 'unknown')
            layer = feat.get('layer', '?')
            print(f"  {i+1}. [L{layer}] {label}")
        
        # Save individual result
        with open(OUTPUT_DIR / f"contrast_{name}.json", "w") as f:
            json.dump(result, f, indent=2)
    else:
        print("FAILED")
    
    time.sleep(1)  # Rate limiting

print(f"\n\nCompleted {len(contrast_results)}/{len(CONTRASTS)} contrasts")

In [ ]:
# Save all contrast results
with open(OUTPUT_DIR / "all_contrast_results.json", "w") as f:
    json.dump(contrast_results, f, indent=2)
print(f"Saved to {OUTPUT_DIR / 'all_contrast_results.json'}")

## Analyze Contrast Results: Find Recurring Features

In [ ]:
def extract_feature_id(feat: Dict) -> str:
    """Get unique feature identifier."""
    return f"L{feat.get('layer', '?')}_{feat.get('index_in_sae', feat.get('id', 'unknown'))}"

def analyze_recurring_features(results: Dict, side: str = "top_to_add", top_n: int = 50) -> pd.DataFrame:
    """Find features that appear across multiple contrasts."""
    
    feature_appearances = defaultdict(list)
    feature_info = {}
    
    for contrast_name, data in results.items():
        result = data["result"]
        features = result.get(side, [])[:top_n]
        
        for rank, feat in enumerate(features):
            fid = extract_feature_id(feat)
            feature_appearances[fid].append({
                "contrast": contrast_name,
                "rank": rank + 1,
                "description": data["description"]
            })
            feature_info[fid] = {
                "label": feat.get('label', 'unknown'),
                "layer": feat.get('layer', '?'),
                "index": feat.get('index_in_sae', '?')
            }
    
    # Build dataframe
    rows = []
    for fid, appearances in feature_appearances.items():
        info = feature_info[fid]
        rows.append({
            "feature_id": fid,
            "label": info["label"],
            "layer": info["layer"],
            "index": info["index"],
            "num_contrasts": len(appearances),
            "avg_rank": sum(a["rank"] for a in appearances) / len(appearances),
            "contrasts": [a["contrast"] for a in appearances]
        })
    
    df = pd.DataFrame(rows)
    df = df.sort_values(["num_contrasts", "avg_rank"], ascending=[False, True])
    return df

# Analyze features that appear for PAIN (top_to_add = higher in dataset_1)
pain_features_df = analyze_recurring_features(contrast_results, "top_to_add", top_n=50)
print("Features recurring across contrasts (higher for pain/dataset_1):")
print(pain_features_df.head(20).to_string())

In [ ]:
# Features that appear in 3+ contrasts are strong candidates
strong_pain_features = pain_features_df[pain_features_df["num_contrasts"] >= 3]
print(f"\n\nSTRONG PAIN FEATURE CANDIDATES ({len(strong_pain_features)} features in 3+ contrasts):")
print("="*80)
for _, row in strong_pain_features.iterrows():
    print(f"\n[Layer {row['layer']}] {row['label']}")
    print(f"  Appears in {row['num_contrasts']} contrasts (avg rank: {row['avg_rank']:.1f})")
    print(f"  Contrasts: {', '.join(row['contrasts'])}")

In [ ]:
# Visualize layer distribution of pain features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Layer distribution for recurring features
recurring = pain_features_df[pain_features_df["num_contrasts"] >= 2]
layer_counts = recurring["layer"].value_counts().sort_index()

axes[0].bar(layer_counts.index.astype(str), layer_counts.values)
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Number of Features")
axes[0].set_title("Layer Distribution of Recurring Pain Features")

# Number of contrasts distribution
contrast_counts = pain_features_df["num_contrasts"].value_counts().sort_index()
axes[1].bar(contrast_counts.index, contrast_counts.values)
axes[1].set_xlabel("Number of Contrasts")
axes[1].set_ylabel("Number of Features")
axes[1].set_title("How Many Contrasts Each Feature Appears In")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "layer_distribution.png", dpi=150)
plt.show()

In [ ]:
# Save recurring features analysis
pain_features_df.to_csv(OUTPUT_DIR / "recurring_pain_features.csv", index=False)
strong_pain_features.to_csv(OUTPUT_DIR / "strong_pain_features.csv", index=False)
print("Saved feature analysis to CSV")

---
# PART 2: Per-Sentence Inspection
Get activations for each sentence to see patterns across categories.

In [ ]:
# Run inspection on all sentences (this will take a while)
# Using mean aggregation first

INSPECT_TOP_K = 50  # Top features per sentence

inspection_results_mean = []

print("Running inspection (mean aggregation) on all sentences...")
for i, sentence in enumerate(sentences_1p):
    if i % 20 == 0:
        print(f"  Processing {i+1}/{len(sentences_1p)}...")
    
    messages = [{"role": "user", "content": sentence["prompt"]}]
    result = run_inspect(messages, aggregation="mean", top_k=INSPECT_TOP_K)
    
    if result:
        inspection_results_mean.append({
            "sentence_idx": i,
            "category": sentence["category"],
            "set": sentence["set"],
            "prompt": sentence["prompt"],
            "features": result.get("features", [])
        })
    
    time.sleep(0.5)  # Rate limiting

print(f"\nCompleted {len(inspection_results_mean)} inspections")

In [ ]:
# Save mean inspection results
with open(OUTPUT_DIR / "inspection_mean_all.json", "w") as f:
    json.dump(inspection_results_mean, f, indent=2)
print("Saved mean inspection results")

In [ ]:
# Now run with attribution at end position (colon token)
# Using attribute endpoint with end_idx to get last token

inspection_results_colon = []

print("Running attribution (colon position) on all sentences...")
for i, sentence in enumerate(sentences_1p):
    if i % 20 == 0:
        print(f"  Processing {i+1}/{len(sentences_1p)}...")
    
    messages = [{"role": "user", "content": sentence["prompt"]}]
    # Use negative index or large number for end - API should handle
    result = run_attribute(messages, top_k=INSPECT_TOP_K)
    
    if result:
        inspection_results_colon.append({
            "sentence_idx": i,
            "category": sentence["category"],
            "set": sentence["set"],
            "prompt": sentence["prompt"],
            "features": result.get("features", [])
        })
    
    time.sleep(0.5)

print(f"\nCompleted {len(inspection_results_colon)} attributions")

In [ ]:
# Save colon inspection results
with open(OUTPUT_DIR / "inspection_colon_all.json", "w") as f:
    json.dump(inspection_results_colon, f, indent=2)
print("Saved colon position inspection results")

## Analyze Per-Sentence Inspections

In [ ]:
def build_feature_by_category_matrix(inspection_results: List[Dict]) -> pd.DataFrame:
    """Build matrix of feature activation frequency by category."""
    
    # Count feature appearances per category
    category_features = defaultdict(lambda: defaultdict(int))
    feature_labels = {}
    
    for item in inspection_results:
        cat = item["category"]
        for feat in item["features"]:
            fid = extract_feature_id(feat)
            category_features[cat][fid] += 1
            feature_labels[fid] = feat.get("label", "unknown")
    
    # Build dataframe
    all_features = set()
    for cat_feats in category_features.values():
        all_features.update(cat_feats.keys())
    
    rows = []
    for fid in all_features:
        row = {"feature_id": fid, "label": feature_labels.get(fid, "unknown")}
        for cat in PAIN_CATEGORIES + CONTROL_CATEGORIES:
            row[cat] = category_features[cat].get(fid, 0)
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Add summary columns
    df["pain_total"] = df[PAIN_CATEGORIES].sum(axis=1)
    df["control_total"] = df[CONTROL_CATEGORIES].sum(axis=1)
    df["pain_minus_control"] = df["pain_total"] - df["control_total"]
    
    return df.sort_values("pain_minus_control", ascending=False)

# Analyze mean inspection
if inspection_results_mean:
    matrix_mean = build_feature_by_category_matrix(inspection_results_mean)
    print("Feature activation by category (mean aggregation):")
    print(f"Total unique features: {len(matrix_mean)}")
    print("\nTop 20 features more active for PAIN than controls:")
    display_cols = ["feature_id", "label", "pain_total", "control_total", "pain_minus_control"]
    print(matrix_mean[display_cols].head(20).to_string())

In [ ]:
# Heatmap of top features by category
if inspection_results_mean:
    top_features = matrix_mean.head(30)
    
    # Prepare data for heatmap
    heatmap_data = top_features[PAIN_CATEGORIES + CONTROL_CATEGORIES].values
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        heatmap_data,
        xticklabels=[CATEGORY_NAMES[c] for c in PAIN_CATEGORIES + CONTROL_CATEGORIES],
        yticklabels=top_features["label"].values,
        cmap="RdYlBu_r",
        ax=ax
    )
    ax.set_title("Top 30 Pain-Associated Features by Category")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "feature_category_heatmap.png", dpi=150)
    plt.show()

In [ ]:
# Compare mean vs colon position results
if inspection_results_mean and inspection_results_colon:
    matrix_colon = build_feature_by_category_matrix(inspection_results_colon)
    
    # Find features that are strong in both
    top_mean = set(matrix_mean.head(50)["feature_id"])
    top_colon = set(matrix_colon.head(50)["feature_id"])
    
    overlap = top_mean & top_colon
    mean_only = top_mean - top_colon
    colon_only = top_colon - top_mean
    
    print(f"Top 50 features comparison:")
    print(f"  Overlap (both methods): {len(overlap)}")
    print(f"  Mean-only: {len(mean_only)}")
    print(f"  Colon-only: {len(colon_only)}")
    
    print(f"\nOverlapping features (strongest candidates):")
    for fid in list(overlap)[:10]:
        label = matrix_mean[matrix_mean["feature_id"] == fid]["label"].values[0]
        print(f"  - {label}")

---
# PART 3: Feature Search
Search for features by semantic description to see what pain-related features exist.

In [ ]:
# Search for various pain-related concepts
SEARCH_QUERIES = [
    "physical pain suffering hurt injury",
    "emotional pain grief sadness heartbreak",
    "social rejection exclusion loneliness ostracism",
    "guilt shame regret moral wrongdoing",
    "confusion frustration cognitive difficulty",
    "fear anxiety threat danger",
    "disgust revulsion nausea",
    "body sensation touch feeling physical",
]

search_results = {}

for query in SEARCH_QUERIES:
    print(f"\nSearching: '{query}'")
    result = search_features(query, top_k=30)
    
    if result:
        search_results[query] = result
        print(f"  Found {len(result.get('features', []))} features")
        for feat in result.get("features", [])[:5]:
            print(f"    - [L{feat.get('layer', '?')}] {feat.get('label', 'unknown')}")
    
    time.sleep(1)

In [ ]:
# Save search results
with open(OUTPUT_DIR / "feature_searches.json", "w") as f:
    json.dump(search_results, f, indent=2)
print("Saved feature search results")

---
# PART 4: Cross-Reference & Summary
Combine all findings to identify the strongest pain feature candidates.

In [ ]:
# Cross-reference: features that appear in
# 1. Multiple contrasts (contrastive analysis)
# 2. More often for pain categories (per-sentence inspection)
# 3. Feature search for pain concepts

def get_search_feature_ids(search_results: Dict) -> set:
    """Extract feature IDs from search results."""
    ids = set()
    for query, result in search_results.items():
        if "pain" in query.lower() or "suffering" in query.lower():
            for feat in result.get("features", []):
                ids.add(extract_feature_id(feat))
    return ids

# Get candidates from each method
contrast_candidates = set(strong_pain_features["feature_id"]) if len(strong_pain_features) > 0 else set()
inspection_candidates = set(matrix_mean.head(50)["feature_id"]) if inspection_results_mean else set()
search_candidates = get_search_feature_ids(search_results) if search_results else set()

print(f"Candidates from contrastive analysis: {len(contrast_candidates)}")
print(f"Candidates from per-sentence inspection: {len(inspection_candidates)}")
print(f"Candidates from feature search: {len(search_candidates)}")

# Find overlap
all_candidates = contrast_candidates | inspection_candidates | search_candidates
multi_method = [fid for fid in all_candidates 
                if sum([fid in contrast_candidates, 
                       fid in inspection_candidates, 
                       fid in search_candidates]) >= 2]

print(f"\nFeatures appearing in 2+ methods: {len(multi_method)}")

In [ ]:
# Final summary
print("\n" + "="*80)
print("FINAL SUMMARY: PAIN FEATURE CANDIDATES")
print("="*80)

# Get labels for multi-method features
for fid in multi_method:
    # Try to get label from various sources
    label = "unknown"
    layer = "?"
    
    if len(pain_features_df) > 0:
        match = pain_features_df[pain_features_df["feature_id"] == fid]
        if len(match) > 0:
            label = match.iloc[0]["label"]
            layer = match.iloc[0]["layer"]
    
    sources = []
    if fid in contrast_candidates:
        sources.append("contrast")
    if fid in inspection_candidates:
        sources.append("inspection")
    if fid in search_candidates:
        sources.append("search")
    
    print(f"\n[Layer {layer}] {label}")
    print(f"  ID: {fid}")
    print(f"  Sources: {', '.join(sources)}")

In [ ]:
# Export final summary
summary = {
    "experiment_info": {
        "model": MODEL,
        "dataset": "S1 - Similar Semantic Structure",
        "total_sentences": len(sentences_1p),
        "pain_categories": PAIN_CATEGORIES,
        "control_categories": CONTROL_CATEGORIES
    },
    "contrast_analysis": {
        "num_contrasts": len(contrast_results),
        "strong_features": len(strong_pain_features) if len(strong_pain_features) > 0 else 0
    },
    "inspection_analysis": {
        "sentences_inspected": len(inspection_results_mean) if inspection_results_mean else 0,
        "unique_features": len(matrix_mean) if inspection_results_mean else 0
    },
    "multi_method_candidates": multi_method
}

with open(OUTPUT_DIR / "experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nAll results saved to: {OUTPUT_DIR.absolute()}")
print("\nFiles:")
for f in OUTPUT_DIR.glob("*"):
    print(f"  - {f.name}")